# <font color="#F48E16">XAI aplicada al Deep Learning: Permutación</font>

Material generado por <a href="https://www.linkedin.com/in/christian-oliva-moya-ingeniero/">Christian Oliva</a>. Cualquier duda, sugerencia o errata, no duden en contactar.

**Versión 1.0** - 28 de abril de 2026

In [ ]:
import tensorflow as tf

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import kagglehub
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from tqdm import tqdm

## <font color="#F48E16">Datos</font>

En este notebook se muestra la implementación manual del algoritmo de importancia por permutación.

Para ello, se van a utilizar diferentes modelos sencillos de SKLearn sobre un dataset de riesgo financiero para la aprobación de préstamos: **Loan Approval Classification Dataset**

https://www.kaggle.com/datasets/taweilo/loan-approval-classification-data

<hr>

En el notebook se desarrolla el código por completo según una fase sencilla de preprocesamiento de los datos, el entrenamiento de algunos modelos y la explicabilidad utilizando los diferentes algoritmos.

### Descarga de datos de Kaggle

In [ ]:
# Download latest version
path = kagglehub.dataset_download("taweilo/loan-approval-classification-data")

print("Path to dataset files:", path)

### Primer vistazo de los datos

In [ ]:
data = pd.read_csv(path + "/loan_data.csv")
data

### Preprocesamiento

#### Person Age

In [ ]:
variable = data["person_age"]
variable.hist(bins=31)

In [ ]:
variable = np.clip(variable, 0, 70)
variable = np.log(variable)
variable.hist(bins=31)

#### Person Income

In [ ]:
variable = data["person_income"]
variable.hist(bins=31)

In [ ]:
np.log(variable).hist(bins=31)

#### Person Emp Exp (años de experiencia)

In [ ]:
variable = data["person_emp_exp"]
variable.hist(bins=31)

In [ ]:
variable = np.log(variable+1) # Sumo 1 para evitar -inf
variable.hist(bins=31)

#### Person Home Ownership (situación de propiedad de la vivienda)

Los únicos valores posibles son RENT (alquiler), OWN (propiedad), MORTGAGE (hipoteca) y OTHER (otro).

In [ ]:
variable = data["person_home_ownership"]
variable.unique()

In [ ]:
pd.get_dummies(variable, prefix="person_home_ownership_")

#### Loan Amnt (valor del préstamo)

In [ ]:
variable = data["loan_amnt"]
variable.hist(bins=31)

In [ ]:
variable = np.log(variable)
variable.hist(bins=31)

#### Loan Intent (finalidad del préstamo)

Tiene un conjunto finito de valores: PERSONAL (personal), EDUCATION (educación), VENTURE (médico), HOMEIMPROVEMENT (mejora del hogar) y DEBTCONSOLIDATION (consolidación de deudas).

In [ ]:
variable = data["loan_intent"]
variable.unique()

In [ ]:
pd.get_dummies(variable, prefix="loan_intent_")

#### Loan Int Rate (tipo de interés)

In [ ]:
variable = data["loan_int_rate"]
variable.hist(bins=31)

#### Loan Percent Income (pct del préstamo respecto a los ingresos anuales)

In [ ]:
variable = data["loan_percent_income"]
variable.hist(bins=31)

In [ ]:
variable = np.log(variable+1e-1)
variable.hist(bins=31)

#### Cb Person Cred Hist Length (duración del crédito)

In [ ]:
variable = data["cb_person_cred_hist_length"]
variable.hist(bins=31)

In [ ]:
variable = np.log(variable)
variable.hist(bins=31)

#### Credit Score

In [ ]:
variable = data["credit_score"]
variable.hist(bins=31)

#### Previous Loan Defaults on File (impagos anteriores)

In [ ]:
variable = data["previous_loan_defaults_on_file"]
variable.unique()

In [ ]:
variable.replace({"No": 0, "Yes": 1})

#### TODO JUNTO

In [ ]:
data["person_age"] = np.log(np.clip(data["person_age"], 0, 70))
data["person_gender"] = data["person_gender"].replace({"female":0, "male":1})
data["person_education"] = data["person_education"].replace({"High School":0, "Associate":1, "Bachelor":2, "Master":3, "Doctorate":4})
data["person_income"] = np.log(data["person_income"])
data["person_emp_exp"] = np.log(data["person_emp_exp"]+1)
data = pd.concat((data, pd.get_dummies(data["person_home_ownership"], prefix="person_home_ownership_")), axis=1)
data = data.drop(columns=["person_home_ownership"])
data["loan_amnt"] = np.log(data["loan_amnt"])
data = pd.concat((data, pd.get_dummies(data["loan_intent"], prefix="loan_intent_")), axis=1)
data = data.drop(columns=["loan_intent"])
data["loan_percent_income"] = np.log(data["loan_percent_income"]+0.1)
data["cb_person_cred_hist_length"] = np.log(data["cb_person_cred_hist_length"])
data["previous_loan_defaults_on_file"] = data["previous_loan_defaults_on_file"].replace({"No": 0, "Yes": 1})

### Separación en TRAIN-TEST y normalización

In [ ]:
X = data.drop(columns=["loan_status"])
y = data["loan_status"]

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
means = X_train_raw.mean()
stds = X_train_raw.std()
X_train = (X_train_raw-means) / stds
X_test = (X_test_raw-means) / stds

In [ ]:
columnas = X_train_raw.columns
columnas

In [ ]:
X_train = X_train.values
y_train = y_train.values
X_test = X_test.values
y_test = y_test.values

## <font color="#F48E16">Modelos</font>

### Regresión Logística (modelo lineal)

In [ ]:
rl = LogisticRegression()
rl.fit(X_train, y_train)
pred = rl.predict(X_test)
rl.score(X_test, y_test)

In [ ]:
sns.heatmap(confusion_matrix(y_test, pred, normalize="true"), annot=True)
plt.ylabel("Real")
plt.xlabel("Predicción")
plt.show()

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel().tolist()

print(" > ACCURACY:", (tp+tn)/(tp+tn+fp+fn))
print(" > PRECISION:", tp/(tp+fp))
print(" > RECALL:", tp/(tp+fn))

### SVM

In [ ]:
svc = SVC(kernel="rbf", C=1.0, gamma="scale", max_iter=2000, probability=True)
svc.fit(X_train, y_train)
pred = svc.predict(X_test)
svc.score(X_test, y_test)

In [ ]:
sns.heatmap(confusion_matrix(y_test, pred, normalize="true"), annot=True)
plt.ylabel("Real")
plt.xlabel("Predicción")
plt.show()

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel().tolist()

print(" > ACCURACY:", (tp+tn)/(tp+tn+fp+fn))
print(" > PRECISION:", tp/(tp+fp))
print(" > RECALL:", tp/(tp+fn))

### MLP

In [ ]:
red = MLPClassifier(hidden_layer_sizes=(100,), max_iter=2000)
red.fit(X_train, y_train)
pred = red.predict(X_test)
red.score(X_test, y_test)

In [ ]:
sns.heatmap(confusion_matrix(y_test, pred, normalize="true"), annot=True)
plt.ylabel("Real")
plt.xlabel("Predicción")
plt.show()

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel().tolist()

print(" > ACCURACY:", (tp+tn)/(tp+tn+fp+fn))
print(" > PRECISION:", tp/(tp+fp))
print(" > RECALL:", tp/(tp+fn))

## <font color="#F48E16">Explicabilidad</font>

**IMPORTANTE**: Por simplicidad para los algoritmos más pesados, vamos a utilizar solo un trocito de X_train como si fuese nuestro dataset completo.

In [ ]:
X_train_small = X_train[:100]
y_train_small = y_train[:100]

### Importancia por permutación [Método global]

La importancia por permutación consiste en permutar uno a uno los atributos del dataset de entrada y evaluar el modelo M para ver cómo se modifica su rendimiento. La relevancia, por tanto, se define así:

$$R_i = M(X) - \frac{1}{N} \sum_{j=1}^N M(X'_i)$$

donde $R_i$ es la relevancia del atributo $i$-ésimo, $M(X)$ representa la métrica de rendimiento del modelo $M$ a partir del dataset original $X$, $N$ es el número de repeticiones para promediar, y $X'_i$ hace referencia al dataset original X habiendo permutado el atributo $i$-ésimo.

**¿Por qué se hace esto?**

Porque si se pierde rendimiento cuando se rompe el atributo $i$-ésimo, esto significa que dicho atributo es importante para el modelo.

Este algoritmo de explicabilidad es un método global, ya que utiliza todo el dataset para dar un valor global de la relevancia de un atributo.

In [ ]:
def importancia_permutacion(X, y, model, N=10):
    Rx = np.zeros(X.shape[1])

    ##################################
    # TO-DO Implementa el algoritmo de importancia por permutación
    ##################################

    return Rx

In [ ]:
Rx_perm_rl = importancia_permutacion(X_train_small, y_train_small, rl, N=10)
Rx_perm_svc = importancia_permutacion(X_train_small, y_train_small, svc, N=10)
Rx_perm_red = importancia_permutacion(X_train_small, y_train_small, red, N=10)

In [ ]:
plt.figure(figsize=(10, 3))
plt.bar(columnas, Rx_perm_rl)
plt.xticks(rotation=90)
plt.grid(alpha=0.2)
plt.axhline(0.0, color="red")
plt.show()

In [ ]:
plt.figure(figsize=(10, 3))
plt.bar(columnas, Rx_perm_svc)
plt.xticks(rotation=90)
plt.grid(alpha=0.2)
plt.show()

In [ ]:
plt.figure(figsize=(10, 3))
plt.bar(columnas, Rx_perm_red)
plt.xticks(rotation=90)
plt.grid(alpha=0.2)
plt.show()

In [ ]:
x = np.arange(len(columnas))
width = 0.25

plt.figure(figsize=(10, 3))
plt.bar(x - width, np.abs(Rx_perm_rl), width, label="Regresión Logística")
plt.bar(x, np.abs(Rx_perm_svc), width, label="SVC")
plt.bar(x + width, np.abs(Rx_perm_red), width, label="Red Neuronal")
plt.xticks(x, columnas, rotation=90)
plt.ylabel("Importancia por Permutación")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

**¿Conclusiones?**

Yo diría que...

- Si para 3 modelos distintos hay atributos que tienen siempre relevancia con magnitud bajita, es que esos atributos sobran en el dataset.

- Si hay algunos atributos que siempre tienen relevancia alta, ya sea por arriba o por abajo, es que son relevantes en el dataset.

- La clave está en que algunos atributos son útiles para algún modelo pero inútiles para otro. Eso significa que estás explicando el modelo en particular y no seleccionando atributos en general.